# CGLOPS Biophysical Products Explorer

This notebook demonstrates how to discover, search, load, and visualise **CGLOPS biophysical products** from the [Copernicus Data Space (CDSE)](https://dataspace.copernicus.eu/) using the `rs_tools` package.

Products are accessed via the **OData API** (search) and **S3 object storage** (COG streaming).  The pipeline automatically clips global rasters to your bounding box — only the tiles overlapping the AOI are fetched.

**Products demonstrated:**
- NDVI 300 m (10-daily)
- LAI 300 m (10-daily)
- FAPAR 300 m (10-daily)
- GPP / NPP 300 m (10-daily)

We will:
1. List known CLMS products from the built-in catalog
2. Search CDSE and load real raster data for a small AOI
3. Visualise seasonal time-series with interactive slider
4. Compare complementary products with slider and RGB composites

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from rs_tools.config import BoundingBox
from rs_tools.datasets.catalog import list_datasets, get
from rs_tools.datasets.loader import load_dataset, items_to_dataarray
from rs_tools.visualization.timeseries import plot_timeseries_slider, plot_timeseries_line
from rs_tools.visualization.globe import add_globe_inset
from rs_tools.visualization.slider import slider_comparison
from rs_tools.visualization.rgb_composite import multi_temporal_rgb, plot_rgb

%matplotlib widget

## Known CLMS/CGLOPS Products

List all biophysical products registered in the catalog.

In [ ]:
clms_datasets = list_datasets(tag="clms")
for ds in clms_datasets:
    res = ds.spatial_resolution or ""
    freq = ds.temporal_resolution or ""
    archives = ", ".join(ds.archive_collections.keys())
    print(f"{ds.short_name:25s} {res:>8s}  {freq:>10s}  archives: {archives}")

## Define Area of Interest

Set up a region of interest and temporal window.  CGLOPS products are global but the loader automatically clips to the bounding box via COG range requests.

In [ ]:
# Belgium / Benelux region — small AOI for fast downloads
bbox = BoundingBox(west=3.0, south=50.0, east=6.0, north=51.5)

START_DATE = "2022-01-01"
END_DATE = "2022-12-31"

print(f"AOI:         {bbox}")
print(f"Time range:  {START_DATE} → {END_DATE}")

## Load NDVI — Vegetation Index

Load CLMS NDVI v3 (300 m, 10-daily) via the OData + S3 pipeline.  Each global COG is ~120 960 × 47 040 pixels but only the tiles intersecting the bounding box are fetched.

In [ ]:
ndvi_items = load_dataset(
    "CLMS_NDVI_V3",
    bbox=bbox,
    start_date=START_DATE,
    end_date=END_DATE,
    limit=40,
)
print(f"Loaded {len(ndvi_items)} NDVI dekads")
for it in ndvi_items[:3]:
    print(f"  {it.label}  shape={next(iter(it.data.values())).shape}")

## Load LAI and FAPAR — Vegetation Properties

Load two complementary vegetation property products for the same region and period.

In [ ]:
lai_items = load_dataset(
    "CLMS_LAI_V2",
    bbox=bbox,
    start_date=START_DATE,
    end_date=END_DATE,
    limit=40,
)
print(f"Loaded {len(lai_items)} LAI dekads")

fapar_items = load_dataset(
    "CLMS_FAPAR_V2",
    bbox=bbox,
    start_date=START_DATE,
    end_date=END_DATE,
    limit=40,
)
print(f"Loaded {len(fapar_items)} FAPAR dekads")

## Visualization: NDVI Breathing Pulse

Animate the seasonal "breathing pulse" of vegetation using the real NDVI time-series.  The slider lets you scrub through dekads interactively.

In [ ]:
ndvi = items_to_dataarray(ndvi_items)
print(f"NDVI shape: {ndvi.sizes}")

fig = plot_timeseries_slider(ndvi, title="NDVI — Belgium 2022", cmap="YlGn", vmin=0, vmax=0.9)
plt.show()

## Globe Inset

Add a 3-D orthographic globe showing the region of interest as context.

In [ ]:
# Show summer NDVI with globe inset
fig, ax = plt.subplots(figsize=(10, 8))
summer_idx = len(ndvi.time) // 2  # mid-year
ax.imshow(ndvi.isel(time=summer_idx).values, cmap="YlGn", vmin=0, vmax=0.9, origin="upper")
ax.set_title(f"NDVI — {str(ndvi.time.values[summer_idx])[:10]}")
add_globe_inset(fig, bbox)
plt.show()

## Product Comparison Slider

Compare NDVI and LAI side-by-side with a draggable divider for the same summer date.

In [ ]:
lai = items_to_dataarray(lai_items)
summer_idx = len(ndvi.time) // 2

left = ndvi.isel(time=summer_idx).values
right = lai.isel(time=min(summer_idx, len(lai.time) - 1)).values

fig = slider_comparison(
    left, right,
    left_label="NDVI",
    right_label="LAI",
    cmap="YlGn",
    title="NDVI vs LAI — Summer 2022",
)
plt.show()

## Multi-Temporal RGB Composite

Assign three dates to R, G, B channels to visualise seasonal change:
- **Red** = winter (January)
- **Green** = spring (April)
- **Blue** = summer (July)

In [ ]:
# Find indices closest to Jan, Apr, Jul
times = ndvi.time.values
targets = [
    np.datetime64("2022-01-15"),
    np.datetime64("2022-04-15"),
    np.datetime64("2022-07-15"),
]
indices = tuple(int(np.argmin(np.abs(times - t))) for t in targets)
print(f"RGB indices: {indices} → {[str(times[i])[:10] for i in indices]}")

rgb = multi_temporal_rgb(ndvi, indices, vmin=0.1, vmax=0.85)
fig = plot_rgb(rgb, title="Multi-Temporal NDVI — R:Winter G:Spring B:Summer")
plt.show()

## Next Steps

- Explore other CLMS products: GPP, NPP, ETA, SWI, Burnt Area
- Run the **7 CGLOPS animation notebooks** in `cglops_animations/` for longer time-series and GIF export
- Add location-specific comparisons for interesting regions (Amazon, Mediterranean, Sahel)
- Experiment with `monthly=True` and `interval_months` for longer time ranges